In [1]:
import pandas as pd

In [2]:
data_inflacion = pd.read_csv("../data/3_BANCO MUNDIAL_INFLACION POR PAISES.csv", skiprows=4)
print(data_inflacion.shape)

(266, 71)


In [3]:
data_inflacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    266 non-null    str    
 1   Country Code    266 non-null    str    
 2   Indicator Name  266 non-null    str    
 3   Indicator Code  266 non-null    str    
 4   1960            70 non-null     float64
 5   1961            72 non-null     float64
 6   1962            74 non-null     float64
 7   1963            75 non-null     float64
 8   1964            79 non-null     float64
 9   1965            86 non-null     float64
 10  1966            94 non-null     float64
 11  1967            101 non-null    float64
 12  1968            102 non-null    float64
 13  1969            103 non-null    float64
 14  1970            109 non-null    float64
 15  1971            113 non-null    float64
 16  1972            115 non-null    float64
 17  1973            118 non-null    float64
 18  1

In [4]:
data_inflacion.head(3)

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,4.270111,NaN
2,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,NaN,NaN


In [5]:
print(data_inflacion.columns.tolist()[:6])
print(data_inflacion.columns.tolist()[-3:])

['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']
['2024', '2025', 'Unnamed: 70']


In [6]:
anios = [str(a) for a in range(2015, 2026)]
print(anios[:3], anios[-1])

['2015', '2016', '2017'] 2025


- `range(2000, 2026)` → números 2000…2025 (el 2026 **no** entra, el final siempre se excluye).
- `str(a)` → convierte cada número a texto, porque los nombres de columna son `"2000"`, no `2000`.
- `[... for a in ...]` → list comprehension: un bucle `for` + `append` en una línea.
- Resultado: `['2000', '2001', '2002'] 2025`.

In [7]:
data_inflacion = data_inflacion[["Country Name", "Country Code"] + anios]
print(data_inflacion.shape)

(266, 13)


- `data_inflacion[[ ... ]]` → **doble corchete** = «quédate solo con estas columnas».
- `["Country Name","Country Code"] + anios` → suma de listas: 2 columnas fijas + las 26 de años = 28.
- Se descartan `Indicator Name`, `Indicator Code`, 1960–1999 y la columna vacía.
- Asignamos de nuevo a `data_inflacion` para «quedarnos» con el resultado.

**Debe salir:** `(266, 28)`.

In [8]:
data_inflacion = data_inflacion.rename(columns={"Country Name": "pais", "Country Code": "iso3"})
print(data_inflacion.columns.tolist()[:3])

['pais', 'iso3', '2015']


- `.rename(columns={"viejo": "nuevo"})` → cambia nombres. Es un método de **DataFrame**, con `columns=` y diccionario.
- Mismo `iso3` en los tres datasets = clave para el merge.

**Debe salir:** `['pais', 'iso3', '2000']`.

## Paso 5 · Quitar los agregados (Mundo, UE, «Ingreso alto»…) con la tabla de equivalencias
El Banco Mundial mezcla países reales con grupos. Nos quedamos solo con códigos que existen en tu tabla ISO.

In [9]:
equivalencias = pd.read_csv("../data/4_EQUIVALENCIAS_PAISES_ISO_3.csv")
print(equivalencias.shape)

(249, 4)


- Leemos tu tabla de países reales (sin `skiprows`: es un CSV normal).

**Debe salir:** `(249, 4)` → columnas `codigo_onu, iso2, iso3, nombre_iso`.

In [10]:
data_inflacion = data_inflacion[data_inflacion["iso3"].isin(equivalencias["iso3"])]
print(data_inflacion.shape)

(215, 13)


- `data_inflacion["iso3"].isin(equivalencias["iso3"])` → para cada fila, `True` si su código está en la tabla de países reales, `False` si no (agregados).
- `data_inflacion[ máscara ]` → deja solo las filas `True`.
- Es el mismo patrón que `[df["col"] == valor]` del filtro de turismo, pero con «está en la lista».

**Debe salir:** `(215, 28)` → se van 51 filas (agregados y territorios que no están en tu tabla).

## Paso 6 · Pasar de formato ancho a largo con `melt`
Ahora mismo hay una columna por año. Necesitamos **una fila por país y año**.

In [11]:
data_inflacion = data_inflacion.melt(id_vars=["pais", "iso3"], var_name="anio", value_name="inflacion_(%)")
print(data_inflacion.shape)
data_inflacion.head(3)

(2365, 4)


,pais,iso3,anio,inflacion_(%)
0,Aruba,ABW,2015,0.474764
1,Afghanistan,AFG,2015,-0.661709
2,Angola,AGO,2015,9.355972


- `.melt(...)` → «derrite» las columnas de años en filas.
- `id_vars=["pais","iso3"]` → columnas que **se quedan fijas** y se repiten.
- `var_name="anio"` → nombre de la nueva columna que guarda el **nombre** de las columnas viejas (2000, 2001…).
- `value_name="inflacion_pct"` → nombre de la columna que guarda los **valores**.
- Analogía: de una tabla de doble entrada (país × año) a una lista «país, año, dato».

**Debe salir:** `(5590, 4)` = 215 países × 26 años. Primera fila: Aruba, ABW, "2000", valor.

## Paso 7 · Convertir `anio` a número entero

In [12]:
data_inflacion["anio"] = data_inflacion["anio"].astype(int)
print(data_inflacion.dtypes)

pais                 str
iso3                 str
anio               int64
inflacion_(%)    float64
dtype: object


- Tras `melt`, `anio` es texto. Con texto, `2000 == "2000"` es `False` y el merge con turismo/vivienda (donde `anio` es `int`) **no casaría**.
- `.astype(int)` convierte la columna a entero.

**Debe salir:** `pais` y `iso3` como `object`/`str`, `anio` `int64`, `inflacion_pct` `float64`.

## Paso 8 · Mirar y tratar los NaN

In [13]:
print(data_inflacion["inflacion_(%)"].isnull().sum())

386


In [15]:
data_inflacion = data_inflacion.dropna(subset=["inflacion_(%)"])
print(data_inflacion.shape)

(1979, 4)


- `.isnull().sum()` → cuántos huecos hay. **Debe salir `916`.**
- `dropna(subset=["inflacion_pct"])` → elimina filas cuyo valor de inflación está vacío. `subset` = «mira solo esta columna».
- Sin dato no hay nada que comparar; no se rellena con 0 (inventaría información).

**Debe salir:** `(4674, 4)`.

## Paso 9 · Ordenar y reiniciar el índice

In [ ]:
data_inflacion = data_inflacion.sort_values(["pais", "anio"]).reset_index(drop=True)
data_inflacion.head()

,pais,iso3,anio,inflacion_pct
0,Afghanistan,AFG,2015,-0.661709
1,Afghanistan,AFG,2016,4.383892
2,Afghanistan,AFG,2017,4.975952
3,Afghanistan,AFG,2018,0.626149
4,Afghanistan,AFG,2019,2.302373


- `sort_values(["pais","anio"])` → ordena por país y, dentro de cada país, por año. Imprescindible para calcular variaciones por país después.
- `reset_index(drop=True)` → renumera filas 0,1,2… y descarta el índice antiguo (que quedó desordenado/con huecos).

**Debe salir:** empieza por Afghanistan 2005 (12.686…): no hay datos antes.

## Paso 10 · Comprobaciones finales

In [ ]:
print(data_inflacion.shape)
print(data_inflacion["iso3"].nunique())
print(data_inflacion.groupby(["iso3", "anio"]).size().max())
data_inflacion[(data_inflacion["iso3"] == "ESP") & (data_inflacion["anio"] >= 2022)]

(1979, 4)
189
1


,pais,iso3,anio,inflacion_pct
1660,Spain,ESP,2022,8.390419
1661,Spain,ESP,2023,3.532473
1662,Spain,ESP,2024,2.774070
1663,Spain,ESP,2025,2.673267


- `nunique()` → cuántos países distintos. **Debe salir `192`.**
- `groupby(["iso3","anio"]).size().max()` → cuenta filas por país-año; el máximo debe ser **1** (sin duplicados). Si sale >1, algo se ha repetido (¿ejecutaste una celda dos veces?).
- Última línea: filtro con dos condiciones, cada una **entre paréntesis** y unidas con `&` (no `and`).

**Debe salir para España:** 2022 → 8.39 · 2023 → 3.53 · 2024 → 2.77 · 2025 → 2.67.

## Paso 11 · Guardar el resultado limpio

In [ ]:
data_inflacion.to_csv("../data/limpio_inflacion.csv", index=False)

- `to_csv(ruta, index=False)` → guarda el DataFrame como CSV. `index=False` evita una columna extra con 0,1,2…
- Así el notebook del merge solo hace `pd.read_csv("../data/limpio_inflacion.csv")` sin repetir la limpieza.

## Resumen de la lógica (para reconocer problemas parecidos)
`leer (skiprows) → quedarse con columnas → renombrar → filtrar agregados (isin) → melt → tipos (astype) → NaN (dropna) → ordenar/reset_index → comprobar → guardar`

Esta secuencia vale para **cualquier** fichero del Banco Mundial en formato ancho.